In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

benefits_tbl = "capstone.bronze.insurance_benefits_raw"
plans_tbl    = "capstone.bronze.insurance_plan_attributes_raw"
silver_tbl   = "capstone.silver.dim_insurance"

b = spark.table(benefits_tbl)
p = spark.table(plans_tbl)

def clean_str(c):
    return F.when(F.trim(F.col(c)) == "", None).otherwise(F.trim(F.col(c)))

# -------------------------
# 1) Build a robust join key (composite)
# -------------------------
def join_key(df):
    return F.concat_ws("|",
        clean_str("BusinessYear"),
        clean_str("StateCode"),
        clean_str("IssuerId"),
        clean_str("StandardComponentId"),
        clean_str("PlanId")
    )

b = b.withColumn("join_key", join_key(b))
p = p.withColumn("join_key", join_key(p))

# -------------------------
# 2) Select/standardize fields from both sides
# -------------------------
# From plans (attributes)
p_sel = (p.select(
    "join_key",
    clean_str("BusinessYear").alias("business_year"),
    clean_str("StateCode").alias("state_code"),
    clean_str("IssuerId").alias("issuer_id"),
    clean_str("StandardComponentId").alias("standard_component_id"),
    clean_str("PlanId").alias("plan_id"),
    clean_str("IssuerMarketPlaceMarketingName").alias("issuer_marketplace_marketing_name"),
    clean_str("PlanMarketingName").alias("plan_marketing_name"),
    clean_str("PlanType").alias("plan_type"),
    clean_str("MetalLevel").alias("metal_level"),
    clean_str("MarketCoverage").alias("market_coverage"),
    clean_str("DentalOnlyPlan").alias("dental_only_plan"),
    clean_str("SourceName").alias("source_name_plan"),
    clean_str("ImportDate").alias("import_date_plan"),
    F.col("ingestion_timestamp").alias("ingestion_ts_plan"),
    clean_str("source_system").alias("source_system_plan")
))

# From benefits (cost sharing)
b_sel = (b.select(
    "join_key",
    clean_str("BenefitName").alias("benefit_name"),
    clean_str("CopayInnTier1").alias("copay_inn_tier1"),
    clean_str("CopayInnTier2").alias("copay_inn_tier2"),
    clean_str("CopayOutofNet").alias("copay_out_of_net"),
    clean_str("CoinsInnTier1").alias("coins_inn_tier1"),
    clean_str("CoinsInnTier2").alias("coins_inn_tier2"),
    clean_str("CoinsOutofNet").alias("coins_out_of_net"),
    clean_str("IsEHB").alias("is_ehb"),
    clean_str("IsCovered").alias("is_covered"),
    clean_str("QuantLimitOnSvc").alias("quant_limit_on_svc"),
    clean_str("LimitQty").alias("limit_qty"),
    clean_str("LimitUnit").alias("limit_unit"),
    clean_str("Exclusions").alias("exclusions"),
    clean_str("Explanation").alias("explanation"),
    clean_str("EHBVarReason").alias("ehb_var_reason"),
    clean_str("IsExclFromInnMOOP").alias("is_excl_from_inn_moop"),
    clean_str("IsExclFromOonMOOP").alias("is_excl_from_oon_moop"),
    F.col("ingestion_timestamp").alias("ingestion_ts_benefits"),
    clean_str("source_system").alias("source_system_benefits")
))

# -------------------------
# 3) Join (left join: keep all plans)
# -------------------------
df = (p_sel.alias("p")
      .join(b_sel.alias("b"), on="join_key", how="left"))

# -------------------------
# 4) Create insurance_name + coverage_details (1 string)
# -------------------------
df = df.withColumn(
    "insurance_name",
    F.coalesce(F.col("plan_marketing_name"), F.col("issuer_marketplace_marketing_name"))
)

# Build a compact coverage_details string (useful in 1-table dim)
df = df.withColumn(
    "coverage_details",
    F.concat_ws(" | ",
        F.concat(F.lit("benefit="), F.col("benefit_name")),
        F.concat(F.lit("copay_inn_t1="), F.col("copay_inn_tier1")),
        F.concat(F.lit("copay_inn_t2="), F.col("copay_inn_tier2")),
        F.concat(F.lit("copay_oon="), F.col("copay_out_of_net")),
        F.concat(F.lit("coins_inn_t1="), F.col("coins_inn_tier1")),
        F.concat(F.lit("coins_inn_t2="), F.col("coins_inn_tier2")),
        F.concat(F.lit("coins_oon="), F.col("coins_out_of_net")),
        F.concat(F.lit("is_ehb="), F.col("is_ehb")),
        F.concat(F.lit("is_covered="), F.col("is_covered")),
        F.concat(F.lit("limit="), F.col("limit_qty"), F.lit(" "), F.col("limit_unit"))
    )
)

# -------------------------
# 5) Stable insurance_id + choose latest ingestion
# -------------------------
natural_key = F.concat_ws("|", F.col("standard_component_id"), F.col("plan_id"), F.col("business_year"))
df = df.withColumn("insurance_id", F.sha2(natural_key, 256))

df = df.withColumn(
    "ingestion_timestamp",
    F.coalesce(F.col("ingestion_ts_plan"), F.col("ingestion_ts_benefits"))
).withColumn(
    "source_system",
    F.coalesce(F.col("source_system_plan"), F.col("source_system_benefits"))
)

# -------------------------
# 6) Deduplicate (keep latest per insurance_id)
# -------------------------
w = Window.partitionBy("insurance_id").orderBy(F.col("ingestion_timestamp").desc_nulls_last())
df = (df
      .withColumn("rn", F.row_number().over(w))
      .filter(F.col("rn") == 1)
      .drop("rn"))

# -------------------------
# 7) Final Silver DIM table (1 table)
# -------------------------
silver_df = df.select(
    "insurance_id",
    "business_year",
    "state_code",
    "issuer_id",
    "standard_component_id",
    "plan_id",
    "insurance_name",
    "plan_type",
    "metal_level",
    "market_coverage",
    "dental_only_plan",
    "coverage_details",
    "ingestion_timestamp",
    "source_system"
)

spark.sql("CREATE SCHEMA IF NOT EXISTS capstone.silver")

(silver_df
 .write
 .mode("overwrite")
 .format("delta")
 .saveAsTable(silver_tbl)
)

display(spark.table(silver_tbl).limit(20))

In [0]:
%sql
SELECT 'Total Rows' AS check_type, COUNT(*) AS result
FROM capstone.silver.dim_insurance

UNION ALL
SELECT 'Missing insurance_id', COUNT(*)
FROM capstone.silver.dim_insurance
WHERE insurance_id IS NULL

UNION ALL
SELECT 'Missing plan_id', COUNT(*)
FROM capstone.silver.dim_insurance
WHERE plan_id IS NULL

UNION ALL
SELECT 'Missing standard_component_id', COUNT(*)
FROM capstone.silver.dim_insurance
WHERE standard_component_id IS NULL

UNION ALL
SELECT 'Missing insurance_name', COUNT(*)
FROM capstone.silver.dim_insurance
WHERE insurance_name IS NULL

UNION ALL
SELECT 'Duplicates insurance_id', COUNT(*)
FROM (
  SELECT insurance_id
  FROM capstone.silver.dim_insurance
  GROUP BY insurance_id
  HAVING COUNT(*) > 1
)

UNION ALL
SELECT 'Missing ingestion_timestamp', COUNT(*)
FROM capstone.silver.dim_insurance
WHERE ingestion_timestamp IS NULL;